In [1]:
from pypdf import PdfReader
from langchain_core.documents import Document

## loader

In [3]:
# pyright: strict
from typing import Iterator


def load_pdf(path: str,start_page:int = 0) ->Iterator[Document] :
    reader = PdfReader(path)
    for i in range(start_page,len(reader.pages)):
        yield Document(
            page_content= reader.pages[i].extract_text().rstrip(),
            metadata= {"source":path, "page":i, "page_label": reader.page_labels[i],  "total_pages": len(reader.pages) }
        )

In [4]:
docs = list(load_pdf("res//PYTHON PROGRAMMING NOTES.pdf",start_page=5))

In [15]:
p38 = next(d for d in docs if d.metadata["page"] == 37)
p38.page_content

'PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET \n33 \n \nC:/Users/MRCET/AppData/Local/Programs/Python/Python38-32/pyyy/fu1.py \nEnter a value 5 \nEnter b value-2 \nbig number=  5 \n#program to find biggest of two numbers using functions. (nested if) \n \ndef biggest(a,b,c): \n    if a>b : \n        if a>c : \n            return a \n        else : \n            return c \n    else : \n        if b>c : \n            return b \n        else : \n            return c \n   \n     \na=int(input("Enter a value")) \nb=int(input("Enter b value")) \nc=int(input("Enter c value")) \n #function call      \nbig= biggest(a,b,c) \nprint("big number= ",big) \n \nOutput: \nC:/Users/MRCET/AppData/Local/Programs/Python/Python38-32/pyyy/fu1.py \nEnter a value 5 \nEnter b value -6 \nEnter c value 7 \nbig number=  7 \n \n#Writer a program to read one subject mark and print pass or fail use single return \nvalues function with argument. \ndef result(a): \n    if

## cleaning headers and footers

In [6]:
import re
from collections import Counter

ZONE = 2          # only look at the first/last 2 non-blank lines of each page
THRESHOLD = 0.6   # a line is boilerplate if it sits in the zone on >= 60% of pages
PAGE_NUM = re.compile(r"\d{1,3}")


def norm(line) -> str:
    return " ".join(line.split())          # collapse runs of spaces so layout jitter doesn't matter


def zone_lines(text: str) -> list[str]:
    lines = [l for l in text.splitlines() if l.strip()]
    return lines[:ZONE] + lines[-ZONE:]

def find_boilerplate(docs: list[Document])->set[str]:
    counts: Counter[str] = Counter()
    for d in docs:
        counts.update({norm(l) for l in zone_lines(d.page_content)})
    return {line for line, n in counts.items() if n / len(docs) >= THRESHOLD}

def clean_page(doc, boilerplate) -> Document:
    lines = doc.page_content.splitlines()
    nonblank = [i for i, l in enumerate(lines) if l.strip()]
    zone = set(nonblank[:ZONE] + nonblank[-ZONE:])
    kept = []
    for i, line in enumerate(lines):
        if i in zone and norm(line) in boilerplate:
            continue
        if i in zone and PAGE_NUM.fullmatch(line.strip()):
            doc.metadata["printed_page"] = int(line.strip())   # keep it as metadata, not text
            continue
        kept.append(line)
    while kept and not kept[0].strip():    # drop whitespace-only lines at the edges,
        kept.pop(0)                         # but never the indentation of a real line
    while kept and not kept[-1].strip():
        kept.pop()
    doc.page_content = "\n".join(kept)
    return doc

In [9]:
boilerplate = find_boilerplate(docs)
print("boilerplate found:", boilerplate)
before = sum(len(d.page_content) for d in docs)
docs = [clean_page(d, boilerplate) for d in docs]
after = sum(len(d.page_content) for d in docs)
print(f"chars removed: {before - after:,} of {before:,}")
print("pages with a printed_page:", sum("printed_page" in d.metadata for d in docs), "/", len(docs))

boilerplate found: set()
chars removed: 18 of 119,530
pages with a printed_page: 137 / 137
